In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np

mpl.rcParams.update(
    {
        "text.usetex": False,
        "axes.labelsize": 20,
        "figure.labelsize": 18,
        "xtick.labelsize": 16,
        "ytick.labelsize": 16,
        "figure.constrained_layout.wspace": 0,
        "figure.constrained_layout.hspace": 0,
        "figure.constrained_layout.h_pad": 0,
        "figure.constrained_layout.w_pad": 0,
        "axes.linewidth": 1.2,
    }
)

import jax
import jax.flatten_util
import jax.numpy as jnp

jax.config.update("jax_enable_x64", True)

## Transfer Functions and Laguerre Quasiseparable Approximation

This notebook shows how to:

1. convolve a stationary GP kernel with a transfer function,
2. wrap the resulting dense kernel with `LaguerreSeries` to obtain a quasiseparable approximation, and
3. use that quasiseparable approximation in the standard `UniVarModel` interface.

The direct convolution is implemented in `eztaox.kernels.transfer_function.ConvolvedKernel` using the Wiener-Khinchin relation,

$$
S_{\rm conv}(f) = S_{\rm base}(f)\,\left|\hat\Psi(f)\right|^2,
$$

followed by an inverse FFT back to the time domain. The Laguerre wrapper then approximates the resulting stationary covariance with a truncated quasiseparable series so that GP inference scales linearly in the number of points.


In [ ]:
from eztaox.kernels.quasisep import Exp, LaguerreSeries, Quasisep
from eztaox.kernels.transfer_function import (
    ConvolvedKernel,
    CausalGaussianTransferFunction,
    TransferFunction,
)
from eztaox.models import UniVarModel
from eztaox.simulator import UniVarSim
from eztaox.ts_utils import add_noise

In [ ]:
class CausalTopHatTransferFunction(TransferFunction):
    """A normalized causal top-hat transfer function."""

    def evaluate(self, X1, X2):
        dt = X2 - X1 - self.shift
        half_width = self.width / 2.0
        inside = (dt >= -half_width) & (dt <= half_width)
        return jnp.where(inside, 1.0 / jnp.maximum(self.width, 1e-12), 0.0)

In [ ]:
import equinox as eqx
import jax
import jax.numpy as jnp
import numpy as np
from scipy.optimize import nnls
from tinygp.kernels import Kernel

from eztaox.kernels.eqx_utils import find_param_by_name
from eztaox.kernels.quasisep import Quasisep


class ExponentialSeries(Quasisep):
    """Approximate a stationary kernel as a nonnegative sum of exponentials.

    The approximation is

        k(tau) ~= sum_i w_i exp(-|tau| / s_i)

    with w_i >= 0, so the resulting kernel is positive semidefinite and
    quasiseparable/state-space friendly.
    """

    scales: jax.Array
    weights: jax.Array

    @classmethod
    def from_kernel(
        cls,
        kernel: Kernel,
        n_terms: int = 12,
        n_fit: int = 512,
        tau_max: float | None = None,
        scale_min_ratio: float = 0.03,
        scale_max_ratio: float = 20.0,
        weight_floor_ratio: float = 1e-3,
        zero_lag_boost: float = 20.0,
    ) -> "ExponentialSeries":
        """Fit a nonnegative exponential mixture to a kernel in the time domain.

        The fit is done using weighted NNLS, approximately minimizing relative
        rather than absolute error across the lag range.
        """
        kernel_scales = find_param_by_name(kernel, "scale")
        if kernel_scales is None:
            raise ValueError(
                "Kernel must have a 'scale' parameter for auto scale placement."
            )

        base_scale = float(sum(kernel_scales) / len(kernel_scales))
        if tau_max is None:
            tau_max = scale_max_ratio * base_scale

        tau_min = max(scale_min_ratio * base_scale * 1e-2, 1e-6)
        tau_fit = np.concatenate([[0.0], np.geomspace(tau_min, tau_max, n_fit - 1)])

        scales = np.geomspace(
            max(scale_min_ratio * base_scale, 1e-6),
            max(scale_max_ratio * base_scale, scale_min_ratio * base_scale * 1.01),
            n_terms,
        )

        k_fit = np.asarray(
            jax.vmap(lambda tau: kernel.evaluate(jnp.array(0.0), jnp.array(tau)))(
                jnp.asarray(tau_fit)
            )
        )
        A = np.exp(-tau_fit[:, None] / scales[None, :])

        k0 = float(kernel.evaluate(jnp.array(0.0), jnp.array(0.0)))
        sigma = np.maximum(np.abs(k_fit), weight_floor_ratio * max(k0, 1e-12))
        sigma[0] /= zero_lag_boost

        Aw = A / sigma[:, None]
        bw = k_fit / sigma
        weights, _ = nnls(Aw, bw)

        wsum = weights.sum()
        if wsum > 0:
            weights *= k0 / wsum

        return cls(scales=jnp.asarray(scales), weights=jnp.asarray(weights))

    def coord_to_sortable(self, X):  # noqa: D102
        return X

    def design_matrix(self) -> jax.Array:  # noqa: D102
        return -jnp.diag(1.0 / self.scales)

    def stationary_covariance(self) -> jax.Array:  # noqa: D102
        return jnp.diag(self.weights)

    def observation_model(self, X) -> jax.Array:  # noqa: D102
        del X
        return jnp.ones_like(self.scales)

    def transition_matrix(self, X1, X2) -> jax.Array:  # noqa: D102
        dt = X2 - X1
        return jnp.diag(jnp.exp(-dt / self.scales))

    def power(
        self, f: float | jax.Array, df: float | jax.Array | None = None
    ) -> jax.Array:
        """PSD of the exponential mixture."""
        del df
        a0 = 1.0 / self.scales
        num = 2.0 * self.weights * a0
        den = a0**2 + (2.0 * jnp.pi * f) ** 2
        return jnp.sum(num / den, axis=0)

### 1. Build a convolved kernel

We start from a DRW/OU kernel and convolve it with a transfer function. The transfer function is normalized internally so that its integral is unity.

Any custom transfer function can be implemented by subclassing `TransferFunction` and defining an `evaluate(X1, X2)` method. Below we compare a built-in causal Gaussian transfer function and a custom causal top-hat transfer function.


In [ ]:
base_kernel = Exp(scale=80.0, sigma=0.2)
gaussian_transfer_function = CausalGaussianTransferFunction(width=60.0, shift=30.0)
top_hat_transfer_function = CausalTopHatTransferFunction(width=60.0, shift=30.0)

# Keep the Gaussian example as the main kernel used later in the notebook.
transfer_function = gaussian_transfer_function
convolved_kernel = ConvolvedKernel(
    base_kernel=base_kernel,
    transfer_function=transfer_function,
    n_grid=2048,
)
convolved_kernel_tophat = ConvolvedKernel(
    base_kernel=base_kernel,
    transfer_function=top_hat_transfer_function,
    n_grid=2048,
)

tau_grid = jnp.linspace(0.0, 300.0, 600)
psi_grid = transfer_function.evaluate(jnp.zeros_like(tau_grid), tau_grid)
psi_grid_tophat = top_hat_transfer_function.evaluate(jnp.zeros_like(tau_grid), tau_grid)
k_base = jax.vmap(lambda tau: base_kernel.evaluate(0.0, tau))(tau_grid)
k_conv = jax.vmap(lambda tau: convolved_kernel.evaluate(0.0, tau))(tau_grid)
k_conv_tophat = jax.vmap(lambda tau: convolved_kernel_tophat.evaluate(0.0, tau))(
    tau_grid
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)

axes[0].plot(
    np.asarray(tau_grid),
    np.asarray(psi_grid),
    color="tab:purple",
    lw=2,
    label="Causal Gaussian",
)
axes[0].plot(
    np.asarray(tau_grid),
    np.asarray(psi_grid_tophat),
    color="tab:green",
    lw=2,
    ls="--",
    label="Causal top-hat",
)
axes[0].set_xlabel(r"Lag $\Delta t$")
axes[0].set_ylabel(r"$\Psi(\Delta t)$")
axes[0].set_title("Transfer function")
axes[0].grid(alpha=0.25)
axes[0].legend(frameon=False)

axes[1].plot(np.asarray(tau_grid), np.asarray(k_base), label="Base Exp kernel", lw=2)
axes[1].plot(
    np.asarray(tau_grid), np.asarray(k_conv), label="Gaussian-convolved kernel", lw=2
)
axes[1].plot(
    np.asarray(tau_grid),
    np.asarray(k_conv_tophat),
    label="Top-hat-convolved kernel",
    lw=2,
    ls="--",
)
axes[1].set_xlabel(r"Lag $|\Delta t|$")
axes[1].set_ylabel(r"$k(\Delta t)$")
axes[1].set_title("Before and after convolution")
axes[1].grid(alpha=0.25)
axes[1].legend(frameon=False)

### 2. Wrap the convolved kernel with `LaguerreSeries`

`ConvolvedKernel` is a dense kernel. To use the scalable quasiseparable solvers in **EzTaoX**, wrap it with `LaguerreSeries`. The wrapped kernel approximates the same stationary covariance while exposing the `Quasisep` interface.


In [ ]:
exp_series = ExponentialSeries.from_kernel(
    convolved_kernel,
    n_terms=24,
    n_fit=1024,
    scale_min_ratio=0.03,
    scale_max_ratio=30.0,
    weight_floor_ratio=1e-3,
)

k_exp_series = jax.vmap(lambda tau: exp_series.evaluate(0.0, tau))(tau_grid)

frac_err = (k_exp_series - k_conv) / jnp.maximum(jnp.abs(k_conv), 1e-12)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)

axes[0].plot(np.asarray(tau_grid), np.asarray(k_conv), label="Gaussian direct", lw=2)
axes[0].plot(
    np.asarray(tau_grid),
    np.asarray(k_exp_series),
    label="Gaussian ExponentialSeries",
    lw=2,
    ls="--",
)
axes[0].set_xlabel(r"Lag $|\Delta t|$")
axes[0].set_ylabel(r"$k(\Delta t)$")
axes[0].set_title("Kernel comparison")
axes[0].grid(alpha=0.25)
axes[0].legend(frameon=False)

axes[1].plot(
    np.asarray(tau_grid), np.asarray(frac_err), color="tab:red", lw=2, label="Gaussian"
)
axes[1].axhline(0.0, color="0.2", lw=1)
axes[1].set_xlabel(r"Lag $|\Delta t|$")
axes[1].set_ylabel("Fractional error")
axes[1].set_title("ExponentialSeries approximation error")
axes[1].grid(alpha=0.25)
axes[1].legend(frameon=False)

In [ ]:
laguerre_kernel = LaguerreSeries(kernel=convolved_kernel, order=12, n_quad=128)
laguerre_kernel_tophat = LaguerreSeries(
    kernel=convolved_kernel_tophat, order=12, n_quad=128
)
k_qs = jax.vmap(lambda tau: laguerre_kernel.evaluate(0.0, tau))(tau_grid)
k_qs_tophat = jax.vmap(lambda tau: laguerre_kernel_tophat.evaluate(0.0, tau))(tau_grid)

print("Convolved kernel is quasiseparable:", isinstance(convolved_kernel, Quasisep))
print("Laguerre wrapper is quasiseparable:", isinstance(laguerre_kernel, Quasisep))
print(
    "Top-hat Laguerre wrapper is quasiseparable:",
    isinstance(laguerre_kernel_tophat, Quasisep),
)

flat_qs, _ = jax.flatten_util.ravel_pytree(laguerre_kernel)
flat_qs

In [ ]:
frac_err = (k_qs - k_conv) / jnp.maximum(jnp.abs(k_conv), 1e-12)
frac_err_tophat = (k_qs_tophat - k_conv_tophat) / jnp.maximum(
    jnp.abs(k_conv_tophat), 1e-12
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)

axes[0].plot(np.asarray(tau_grid), np.asarray(k_conv), label="Gaussian direct", lw=2)
axes[0].plot(
    np.asarray(tau_grid),
    np.asarray(k_qs),
    label="Gaussian LaguerreSeries",
    lw=2,
    ls="--",
)
axes[0].plot(
    np.asarray(tau_grid), np.asarray(k_conv_tophat), label="Top-hat direct", lw=2
)
axes[0].plot(
    np.asarray(tau_grid),
    np.asarray(k_qs_tophat),
    label="Top-hat LaguerreSeries",
    lw=2,
    ls="--",
)
axes[0].set_xlabel(r"Lag $|\Delta t|$")
axes[0].set_ylabel(r"$k(\Delta t)$")
axes[0].set_title("Kernel comparison")
axes[0].grid(alpha=0.25)
axes[0].legend(frameon=False)

axes[1].plot(
    np.asarray(tau_grid), np.asarray(frac_err), color="tab:red", lw=2, label="Gaussian"
)
axes[1].plot(
    np.asarray(tau_grid),
    np.asarray(frac_err_tophat),
    color="tab:green",
    lw=2,
    label="Top-hat",
)
axes[1].axhline(0.0, color="0.2", lw=1)
axes[1].set_xlabel(r"Lag $|\Delta t|$")
axes[1].set_ylabel("Fractional error")
axes[1].set_title("Laguerre approximation error")
axes[1].grid(alpha=0.25)
axes[1].legend(frameon=False)

### 3. Use the quasiseparable approximation in `UniVarModel`

Below we simulate a light curve from the dense convolved kernel, then construct a `UniVarModel` using the quasiseparable Laguerre approximation.

Because `MultiVarModel` and `UniVarModel` reconstruct kernels from a flattened pytree of positive parameters, the simplest initialization is to flatten the kernel and use `log_kernel_param = log(flattened_leaves)`. For this example the leaves are `[scale, sigma, width, shift]`.


In [ ]:
flat_conv, _ = jax.flatten_util.ravel_pytree(convolved_kernel)
sim_params = {"log_kernel_param": jnp.log(flat_conv)}

sim = UniVarSim(convolved_kernel, 1.0, 300.0, sim_params, zero_mean=True)
t_obs, y_true = sim.random(60, jax.random.PRNGKey(1), jax.random.PRNGKey(2))
yerr = jnp.full_like(t_obs, 0.03)
y_obs = add_noise(y_true, yerr, jax.random.PRNGKey(3))

fig, ax = plt.subplots(figsize=(8, 4.5), constrained_layout=True)
ax.errorbar(np.asarray(t_obs), np.asarray(y_obs), np.asarray(yerr), fmt=".")
ax.set_xlabel("Time")
ax.set_ylabel("Flux")
ax.set_title("Simulated light curve")
ax.grid(alpha=0.25)

In [ ]:
model = UniVarModel(t_obs, y_obs, yerr, laguerre_kernel, zero_mean=True)
fit_params = {"log_kernel_param": jnp.log(flat_qs)}

print("Initial log probability:", float(model.log_prob(fit_params)))
model

At this point, `model.log_prob`, `model.pred`, and `model.sample` all use the quasiseparable Laguerre approximation internally, so the GP computations scale linearly in the number of data points.

In practice, you would pair this model with a prior over `log_kernel_param` and then use the standard **EzTaoX** fitting tools (`random_search`, `run_mcmc`, or custom NumPyro code) exactly as in the other notebooks.
